# Optimal Transport
## Moving Mass Optimally: From Monge to Sinkhorn

This notebook provides a self-contained introduction to **optimal transport (OT)** -- the mathematical theory of finding the most efficient way to move mass from one distribution to another. We develop the theory from the classical Monge formulation through Kantorovich's relaxation, implement key algorithms from scratch, and demonstrate practical applications.

**What you'll learn:**
1. The Monge problem and why it is hard for discrete measures
2. Kantorovich's relaxation via linear programming
3. Wasserstein distances and their metric properties
4. Kantorovich duality and c-transforms
5. Entropic regularization and the Sinkhorn algorithm
6. Applications: multi-robot task allocation and Wasserstein barycenters

**Prerequisites:** Linear programming basics, probability distributions, matrix algebra.

**References:**
- Peyr\'{e} & Cuturi, *Computational Optimal Transport*, Foundations and Trends in Machine Learning, 2019.
- Villani, *Optimal Transport: Old and New*, Springer, 2009.
- Cuturi, *Sinkhorn Distances: Lightspeed Computation of Optimal Transport*, NeurIPS, 2013.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import linprog, linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import norm

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# Discrete OT problem sizes
N_SOURCE = 5                  # number of source points
N_TARGET = 6                  # number of target points

# Sinkhorn parameters
SINKHORN_MAX_ITER = 1000      # maximum Sinkhorn iterations
SINKHORN_TOL = 1e-9           # convergence tolerance
EPSILON_VALUES = [1.0, 0.1, 0.01, 0.001]  # regularization strengths

# 1D Wasserstein
N_SAMPLES_1D = 200            # samples for 1D distributions

# Robot task allocation
N_ROBOTS = 8                  # number of robots
N_TASKS = 8                   # number of tasks
ARENA_SIZE = 10.0             # size of the arena

# Barycenter
N_SUPPORT = 50                # barycenter support size
BARYCENTER_MAX_ITER = 100     # fixed-point iterations
BARYCENTER_TOL = 1e-6         # convergence tolerance

# Verification thresholds
DUALITY_GAP_TOL = 1e-8
MARGINAL_TOL = 1e-5
WASSERSTEIN_1D_TOL = 1e-4
SYMMETRY_TOL = 1e-4

# Colors
PRIMARY = 'steelblue'
SECONDARY = 'coral'
TERTIARY = 'seagreen'
ACCENT = 'goldenrod'

---
# Section 1: The Monge Problem

## Historical Context

In 1781, Gaspard Monge posed the following problem: given a pile of sand (source) and a hole to fill (target), what is the optimal way to transport the sand to minimize the total effort?

## Formal Statement

Given two probability measures $\mu$ (source) and $\nu$ (target) on spaces $\mathcal{X}$ and $\mathcal{Y}$, find a **transport map** $T: \mathcal{X} \to \mathcal{Y}$ that:

1. **Pushes $\mu$ forward to $\nu$**: the push-forward condition $T_{\#}\mu = \nu$, meaning for any measurable set $B \subseteq \mathcal{Y}$:
$$\mu(T^{-1}(B)) = \nu(B)$$

2. **Minimizes the transport cost:**
$$\boxed{\inf_{T: T_{\#}\mu = \nu} \int_{\mathcal{X}} c(x, T(x)) \, d\mu(x)}$$

where $c(x, y)$ is a ground cost (e.g., $c(x,y) = \|x - y\|^2$).

## Why Monge is Hard

The Monge problem has two fundamental difficulties:

| Issue | Description |
|-------|-------------|
| **Non-convexity** | The set of valid transport maps $\{T : T_{\#}\mu = \nu\}$ is not convex |
| **Non-existence** | For discrete measures, a transport map may not exist (e.g., Dirac to two Diracs) |
| **Mass splitting** | Monge requires each source point to go to *exactly one* target -- no splitting allowed |

In [ ]:
# =============================================================================
# Section 1: Visualize Why Monge Fails for Discrete Measures
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: valid Monge map (n_source = n_target, uniform weights)
ax = axes[0]
src_1d = np.array([1.0, 3.0, 5.0])
tgt_1d = np.array([2.0, 4.0, 7.0])
ax.scatter(src_1d, np.ones(3), s=200, c=PRIMARY, zorder=5, label='Source $\\mu$')
ax.scatter(tgt_1d, np.zeros(3), s=200, c=SECONDARY, zorder=5, label='Target $\\nu$')
for s, t in zip(src_1d, tgt_1d):
    ax.annotate('', xy=(t, 0.05), xytext=(s, 0.95),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.set_ylim(-0.3, 1.5)
ax.set_xlabel('Position')
ax.set_title('Monge Map Exists (equal masses)')
ax.legend(loc='upper left')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Target', 'Source'])

# Right: Monge map does NOT exist (1 source, 2 targets)
ax = axes[1]
ax.scatter([3.0], [1.0], s=400, c=PRIMARY, zorder=5, label='Source $\\mu$ (mass=1)')
ax.scatter([1.0, 5.0], [0.0, 0.0], s=200, c=SECONDARY, zorder=5, label='Target $\\nu$ (mass=0.5 each)')
ax.annotate('$T(x)=?$', xy=(3.0, 0.5), fontsize=14, ha='center', color='red')
ax.annotate('', xy=(1.0, 0.05), xytext=(3.0, 0.95),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='dashed'))
ax.annotate('', xy=(5.0, 0.05), xytext=(3.0, 0.95),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='dashed'))
ax.set_ylim(-0.3, 1.5)
ax.set_xlabel('Position')
ax.set_title('No Monge Map (mass must split)')
ax.legend(loc='upper left')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Target', 'Source'])

plt.tight_layout()
plt.show()

---
# Section 2: Kantorovich Relaxation

## From Maps to Plans

Leonid Kantorovich (1942, Nobel Prize 1975) relaxed Monge's problem by replacing the deterministic transport map $T$ with a **transport plan** (or coupling) $\pi$.

Instead of sending each source point to exactly one target, we allow mass to be **split** across multiple targets. The transport plan $\pi_{ij} \geq 0$ specifies how much mass flows from source $i$ to target $j$.

## Discrete Formulation

Given:
- Source weights $\mathbf{a} \in \mathbb{R}^n$ with $\sum_i a_i = 1$
- Target weights $\mathbf{b} \in \mathbb{R}^m$ with $\sum_j b_j = 1$
- Cost matrix $C \in \mathbb{R}^{n \times m}$ where $C_{ij} = c(x_i, y_j)$

The Kantorovich problem is a **linear program (LP)**:

$$\boxed{\min_{\pi \in \Pi(\mathbf{a}, \mathbf{b})} \langle C, \pi \rangle = \sum_{i,j} C_{ij} \pi_{ij}}$$

where the set of **admissible couplings** is:

$$\Pi(\mathbf{a}, \mathbf{b}) = \left\{ \pi \in \mathbb{R}^{n \times m}_+ \;:\; \pi \mathbf{1}_m = \mathbf{a}, \;\; \pi^\top \mathbf{1}_n = \mathbf{b} \right\}$$

**Key insight:** Unlike Monge, the Kantorovich problem is a convex (in fact linear) program, so it always has a solution. The Birkhoff-von Neumann theorem guarantees that the LP has an optimal vertex solution with at most $n + m - 1$ nonzero entries.

In [ ]:
# =============================================================================
# Section 2: Kantorovich LP Solver
# =============================================================================

def kantorovich_lp(C, a, b):
    """Solve the Kantorovich optimal transport problem via linear programming.

    Solves: min <C, pi>  s.t.  pi @ 1 = a,  pi.T @ 1 = b,  pi >= 0.

    Args:
        C: Cost matrix. Shape: (n, m).
        a: Source distribution. Shape: (n,).
        b: Target distribution. Shape: (m,).

    Returns:
        pi_opt: Optimal transport plan. Shape: (n, m).
        ot_cost: Optimal transport cost. Scalar.
    """
    n, m = C.shape
    assert len(a) == n and len(b) == m
    assert np.abs(np.sum(a) - np.sum(b)) < 1e-10, "Masses must be equal"

    # Flatten cost matrix to vector (row-major)
    c_vec = C.flatten()

    # Equality constraints: A_eq @ pi_vec = b_eq
    # Row marginals: sum over j for each i => pi @ 1 = a
    # Column marginals: sum over i for each j => pi.T @ 1 = b
    A_row = np.zeros((n, n * m))
    for i in range(n):
        A_row[i, i * m:(i + 1) * m] = 1.0

    A_col = np.zeros((m, n * m))
    for j in range(m):
        A_col[j, j::m] = 1.0

    A_eq = np.vstack([A_row, A_col])
    b_eq = np.concatenate([a, b])

    # Solve LP
    result = linprog(
        c_vec,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=[(0, None)] * (n * m),
        method='highs'
    )

    pi_opt = result.x.reshape(n, m)
    ot_cost = result.fun

    return pi_opt, ot_cost

In [ ]:
# =============================================================================
# Setup: Create Source and Target Distributions
# =============================================================================

np.random.seed(42)

# Source points in 2D
x_source = np.array([
    [1.0, 2.0],
    [2.0, 4.0],
    [3.0, 1.0],
    [4.0, 3.5],
    [5.0, 2.5],
])

# Target points in 2D
y_target = np.array([
    [7.0, 1.0],
    [8.0, 3.0],
    [9.0, 2.0],
    [7.5, 4.5],
    [8.5, 0.5],
    [10.0, 3.5],
])

# Non-uniform weights
a = np.array([0.25, 0.15, 0.20, 0.10, 0.30])
b = np.array([0.10, 0.20, 0.15, 0.15, 0.25, 0.15])

# Verify they sum to 1
assert np.isclose(a.sum(), 1.0) and np.isclose(b.sum(), 1.0)

# Cost matrix: squared Euclidean distance
C = cdist(x_source, y_target, metric='sqeuclidean')

print(f"Source distribution a: {a}  (sum = {a.sum():.2f})")
print(f"Target distribution b: {b}  (sum = {b.sum():.2f})")
print(f"Cost matrix C shape: {C.shape}")
print(f"\nCost matrix C:\n{np.array2string(C, precision=2, suppress_small=True)}")

In [ ]:
# =============================================================================
# Solve Kantorovich LP
# =============================================================================

pi_lp, ot_cost_lp = kantorovich_lp(C, a, b)

print(f"Optimal transport cost (LP): {ot_cost_lp:.6f}")
print(f"\nOptimal transport plan pi:")
print(np.array2string(pi_lp, precision=4, suppress_small=True))

# Verify marginals
row_marginal_err = np.max(np.abs(pi_lp.sum(axis=1) - a))
col_marginal_err = np.max(np.abs(pi_lp.sum(axis=0) - b))
print(f"\nRow marginal max error: {row_marginal_err:.2e}")
print(f"Col marginal max error: {col_marginal_err:.2e}")

# Count nonzeros (Birkhoff: at most n + m - 1)
nnz = np.sum(pi_lp > 1e-10)
print(f"Nonzero entries: {nnz} (Birkhoff bound: {N_SOURCE + N_TARGET - 1})")

---
# Section 3: Wasserstein Distance

## Definition

The **$p$-Wasserstein distance** between distributions $\mu$ and $\nu$ is:

$$\boxed{W_p(\mu, \nu) = \left( \min_{\pi \in \Pi(\mu, \nu)} \sum_{i,j} c_{ij}^p \, \pi_{ij} \right)^{1/p}}$$

For $p=1$ with ground metric $c_{ij} = \|x_i - y_j\|$, this is the **Earth Mover's Distance**.

## Metric Properties

The Wasserstein distance satisfies:
1. **Non-negativity:** $W_p(\mu, \nu) \geq 0$
2. **Identity:** $W_p(\mu, \nu) = 0 \iff \mu = \nu$
3. **Symmetry:** $W_p(\mu, \nu) = W_p(\nu, \mu)$
4. **Triangle inequality:** $W_p(\mu, \rho) \leq W_p(\mu, \nu) + W_p(\nu, \rho)$

## 1D Analytical Formula

In one dimension, the Wasserstein distance has a beautiful closed form using **quantile functions** (inverse CDFs):

$$W_p(\mu, \nu) = \left( \int_0^1 |F_\mu^{-1}(t) - F_\nu^{-1}(t)|^p \, dt \right)^{1/p}$$

For $p=1$: $W_1 = \int_{-\infty}^{\infty} |F_\mu(x) - F_\nu(x)| \, dx$.

In [ ]:
# =============================================================================
# Section 3: Wasserstein Distance Implementation
# =============================================================================

def wasserstein_distance(C, a, b, p=1):
    """Compute the p-Wasserstein distance between two discrete distributions.

    Args:
        C: Ground cost matrix (distances, NOT raised to power p). Shape: (n, m).
        a: Source distribution. Shape: (n,).
        b: Target distribution. Shape: (m,).
        p: Wasserstein exponent. Scalar.

    Returns:
        W_p: The p-Wasserstein distance. Scalar.
        pi_opt: Optimal transport plan. Shape: (n, m).
    """
    C_p = C ** p
    pi_opt, ot_cost = kantorovich_lp(C_p, a, b)
    W_p = ot_cost ** (1.0 / p)
    return W_p, pi_opt


def wasserstein_1d(x, a, y, b, p=1):
    """Compute the p-Wasserstein distance for 1D distributions via quantile sorting.

    Uses the closed-form: sort both distributions, align quantiles, compute cost.
    For discrete distributions with equal total mass, this reduces to sorting
    and computing the weighted sum of distances between matched quantiles.

    Args:
        x: Source support points. Shape: (n,).
        a: Source weights. Shape: (n,).
        y: Target support points. Shape: (m,).
        b: Target weights. Shape: (m,).
        p: Wasserstein exponent (default 1). Scalar.

    Returns:
        W_p: The p-Wasserstein distance. Scalar.
    """
    # Build CDFs on a fine grid
    all_pts = np.sort(np.unique(np.concatenate([x, y])))
    # Extend grid slightly
    grid = np.linspace(all_pts[0] - 1, all_pts[-1] + 1, 10000)

    # CDF of source
    idx_x = np.argsort(x)
    x_sorted, a_sorted = x[idx_x], a[idx_x]
    cdf_a = np.zeros_like(grid)
    for xi, ai in zip(x_sorted, a_sorted):
        cdf_a += ai * (grid >= xi).astype(float)

    # CDF of target
    idx_y = np.argsort(y)
    y_sorted, b_sorted = y[idx_y], b[idx_y]
    cdf_b = np.zeros_like(grid)
    for yj, bj in zip(y_sorted, b_sorted):
        cdf_b += bj * (grid >= yj).astype(float)

    # W_p via CDF integration (for p=1, this is exact up to grid resolution)
    dx = grid[1] - grid[0]
    if p == 1:
        W_p = np.sum(np.abs(cdf_a - cdf_b)) * dx
    else:
        # Use quantile matching: invert CDFs
        t = np.linspace(0.001, 0.999, 10000)
        # Inverse CDF via linear interpolation
        # Clamp CDFs to [0,1] and ensure monotonicity
        cdf_a_clean = np.clip(cdf_a, 0, 1)
        cdf_b_clean = np.clip(cdf_b, 0, 1)
        # Quantile function: F^{-1}(t) = inf{x : F(x) >= t}
        q_a = np.interp(t, cdf_a_clean, grid)
        q_b = np.interp(t, cdf_b_clean, grid)
        dt = t[1] - t[0]
        W_p = (np.sum(np.abs(q_a - q_b) ** p) * dt) ** (1.0 / p)

    return W_p

In [ ]:
# =============================================================================
# Verification: 1D Wasserstein vs LP
# =============================================================================

np.random.seed(42)

# Create 1D discrete distributions
n1d = 15
m1d = 20
x_1d = np.sort(np.random.randn(n1d) * 2 + 1)
y_1d = np.sort(np.random.randn(m1d) * 1.5 + 4)
a_1d = np.random.dirichlet(np.ones(n1d))
b_1d = np.random.dirichlet(np.ones(m1d))

# LP-based W1
C_1d = np.abs(x_1d[:, None] - y_1d[None, :])  # |x_i - y_j|
W1_lp, _ = wasserstein_distance(C_1d, a_1d, b_1d, p=1)

# Quantile-based W1
W1_quantile = wasserstein_1d(x_1d, a_1d, y_1d, b_1d, p=1)

rel_err_1d = np.abs(W1_lp - W1_quantile) / W1_lp
status_1d = "PASS" if rel_err_1d < WASSERSTEIN_1D_TOL else "FAIL"
print(f"W1 (LP):       {W1_lp:.6f}")
print(f"W1 (quantile): {W1_quantile:.6f}")
print(f"1D Wasserstein matches quantile formula: max relative error = {rel_err_1d:.2e} [{status_1d}]")

In [ ]:
# =============================================================================
# Visualize 1D distributions and CDFs
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: distributions as stem plots
ax = axes[0]
markerline1, stemlines1, baseline1 = ax.stem(x_1d, a_1d, linefmt='-', markerfmt='o', basefmt=' ')
plt.setp(stemlines1, color=PRIMARY, linewidth=2)
plt.setp(markerline1, color=PRIMARY, markersize=8)
markerline2, stemlines2, baseline2 = ax.stem(y_1d, b_1d, linefmt='-', markerfmt='s', basefmt=' ')
plt.setp(stemlines2, color=SECONDARY, linewidth=2)
plt.setp(markerline2, color=SECONDARY, markersize=8)
ax.set_xlabel('x')
ax.set_ylabel('Weight')
ax.set_title('1D Discrete Distributions')
ax.legend(['Source $\\mu$', '', 'Target $\\nu$'], loc='upper right')

# Middle: CDFs
ax = axes[1]
grid = np.linspace(min(x_1d.min(), y_1d.min()) - 1, max(x_1d.max(), y_1d.max()) + 1, 1000)
cdf_a = np.zeros_like(grid)
for xi, ai in zip(np.sort(x_1d), a_1d[np.argsort(x_1d)]):
    cdf_a += ai * (grid >= xi).astype(float)
cdf_b = np.zeros_like(grid)
for yj, bj in zip(np.sort(y_1d), b_1d[np.argsort(y_1d)]):
    cdf_b += bj * (grid >= yj).astype(float)
ax.plot(grid, cdf_a, color=PRIMARY, linewidth=2, label='$F_\\mu$')
ax.plot(grid, cdf_b, color=SECONDARY, linewidth=2, label='$F_\\nu$')
ax.fill_between(grid, cdf_a, cdf_b, alpha=0.2, color=ACCENT,
                label=f'$W_1$ = area = {W1_lp:.3f}')
ax.set_xlabel('x')
ax.set_ylabel('CDF')
ax.set_title('CDFs and $W_1$ as Area Between Them')
ax.legend()

# Right: metric properties -- triangle inequality
ax = axes[2]
# Create a third distribution
z_1d = np.sort(np.random.randn(12) * 1.0 + 2.5)
c_1d = np.random.dirichlet(np.ones(12))

# Compute pairwise W1
C_xz = np.abs(x_1d[:, None] - z_1d[None, :])
C_zy = np.abs(z_1d[:, None] - y_1d[None, :])
W1_xz, _ = wasserstein_distance(C_xz, a_1d, c_1d, p=1)
W1_zy, _ = wasserstein_distance(C_zy, c_1d, b_1d, p=1)

labels = ['$W_1(\\mu,\\nu)$', '$W_1(\\mu,\\rho)$', '$W_1(\\rho,\\nu)$',
          '$W_1(\\mu,\\rho)+W_1(\\rho,\\nu)$']
values = [W1_lp, W1_xz, W1_zy, W1_xz + W1_zy]
bar_colors = [PRIMARY, TERTIARY, SECONDARY, ACCENT]
ax.bar(labels, values, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Distance')
ax.set_title('Triangle Inequality Verification')
# Check triangle inequality
tri_ok = W1_lp <= W1_xz + W1_zy + 1e-10
ax.set_xlabel(f'Triangle inequality holds: {tri_ok}')

plt.tight_layout()
plt.show()

---
# Section 4: Kantorovich Duality

## Dual Problem

Every LP has a dual. The **Kantorovich dual** of the OT problem is:

$$\boxed{\max_{f, g} \sum_i f_i a_i + \sum_j g_j b_j \quad \text{s.t.} \quad f_i + g_j \leq C_{ij} \;\; \forall i,j}$$

where $f \in \mathbb{R}^n$ and $g \in \mathbb{R}^m$ are the **Kantorovich potentials**.

## Complementary Slackness

At optimality, $\pi^*_{ij} > 0 \implies f^*_i + g^*_j = C_{ij}$. This means mass only flows along "tight" edges where the dual constraint is active.

## c-Transform

Given a potential $f$, its **$c$-transform** is:
$$f^c(y_j) = \min_i \left[ C_{ij} - f_i \right]$$

At optimality, $g^* = f^{*c}$ (the optimal $g$ is the $c$-transform of the optimal $f$).

## Strong Duality

By LP strong duality, the primal and dual optimal values are **equal**:
$$\min_{\pi} \langle C, \pi \rangle = \max_{f,g} \langle f, a \rangle + \langle g, b \rangle$$

The **duality gap** at optimality is exactly zero.

In [ ]:
# =============================================================================
# Section 4: Kantorovich Dual Solver
# =============================================================================

def kantorovich_dual(C, a, b):
    """Solve the Kantorovich dual problem via linear programming.

    Dual: max  f.a + g.b   s.t.  f_i + g_j <= C_ij  for all i,j.
    Rewritten as min: min  -f.a - g.b  s.t.  f_i + g_j <= C_ij.

    Args:
        C: Cost matrix. Shape: (n, m).
        a: Source distribution. Shape: (n,).
        b: Target distribution. Shape: (m,).

    Returns:
        f_opt: Optimal source potential. Shape: (n,).
        g_opt: Optimal target potential. Shape: (m,).
        dual_value: Optimal dual objective. Scalar.
    """
    n, m = C.shape

    # Decision variables: [f_0, ..., f_{n-1}, g_0, ..., g_{m-1}]
    # Objective: min -a^T f - b^T g
    c_obj = np.concatenate([-a, -b])

    # Inequality constraints: f_i + g_j <= C_ij  for all i,j
    # Number of constraints: n * m
    A_ub = np.zeros((n * m, n + m))
    b_ub = np.zeros(n * m)

    idx = 0
    for i in range(n):
        for j in range(m):
            A_ub[idx, i] = 1.0       # f_i
            A_ub[idx, n + j] = 1.0   # g_j
            b_ub[idx] = C[i, j]
            idx += 1

    # No bounds on f, g (they are free variables)
    bounds = [(None, None)] * (n + m)

    result = linprog(
        c_obj,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=bounds,
        method='highs'
    )

    f_opt = result.x[:n]
    g_opt = result.x[n:]
    dual_value = -result.fun  # We minimized the negative

    return f_opt, g_opt, dual_value

In [ ]:
# =============================================================================
# Verification: Primal-Dual Gap
# =============================================================================

# Solve dual
f_opt, g_opt, dual_value = kantorovich_dual(C, a, b)

# Primal value (already computed)
primal_value = ot_cost_lp

duality_gap = np.abs(primal_value - dual_value)
status_gap = "PASS" if duality_gap < DUALITY_GAP_TOL else "FAIL"

print(f"Primal (LP) value:  {primal_value:.10f}")
print(f"Dual value:         {dual_value:.10f}")
print(f"Primal-dual gap: max relative error = {duality_gap:.2e} [{status_gap}]")

# Verify c-transform relationship
g_ctransform = np.array([np.min(C[:, j] - f_opt) for j in range(N_TARGET)])
ctransform_err = np.max(np.abs(g_opt - g_ctransform))
print(f"\nc-transform verification: max |g - f^c| = {ctransform_err:.2e}")

# Verify complementary slackness
slack = C - f_opt[:, None] - g_opt[None, :]
cs_violation = np.max(pi_lp * slack)
print(f"Complementary slackness violation: {cs_violation:.2e}")

In [ ]:
# =============================================================================
# Visualize Dual Potentials and Complementary Slackness
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Kantorovich potentials
ax = axes[0]
x_idx = np.arange(N_SOURCE)
y_idx = np.arange(N_TARGET)
ax.bar(x_idx - 0.15, f_opt, width=0.3, color=PRIMARY, label='$f^*$ (source)')
ax.bar(y_idx + 0.15 + N_SOURCE, g_opt, width=0.3, color=SECONDARY, label='$g^*$ (target)')
ax.set_xlabel('Index')
ax.set_ylabel('Potential value')
ax.set_title('Kantorovich Dual Potentials')
ax.legend()

# Middle: slack matrix
ax = axes[1]
im = ax.imshow(slack, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='$C_{ij} - f_i - g_j$')
ax.set_xlabel('Target j')
ax.set_ylabel('Source i')
ax.set_title('Dual Slack (0 = tight constraint)')

# Right: transport plan with tight edges highlighted
ax = axes[2]
im2 = ax.imshow(pi_lp, cmap='Blues', aspect='auto')
plt.colorbar(im2, ax=ax, label='$\\pi^*_{ij}$')
# Mark tight edges
for i in range(N_SOURCE):
    for j in range(N_TARGET):
        if pi_lp[i, j] > 1e-10:
            ax.text(j, i, f'{pi_lp[i,j]:.3f}', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='red')
ax.set_xlabel('Target j')
ax.set_ylabel('Source i')
ax.set_title('Optimal Transport Plan')

plt.tight_layout()
plt.show()

---
# Section 5: Entropic Regularization and the Sinkhorn Algorithm

## Motivation

The LP-based OT solver has complexity $O(n^3 \log n)$, which becomes prohibitive for large $n$. **Entropic regularization** replaces the LP with a smooth, strictly convex problem that can be solved via matrix scaling.

## Regularized Problem

Add a negative entropy penalty:

$$\min_{\pi \in \Pi(\mathbf{a}, \mathbf{b})} \langle C, \pi \rangle - \varepsilon H(\pi)$$

where $H(\pi) = -\sum_{i,j} \pi_{ij} (\log \pi_{ij} - 1)$ is the (negentropy) entropy.

## KKT Conditions and Sinkhorn Iterations

The KKT conditions for the regularized problem give:

$$\pi^*_{ij} = u_i \, K_{ij} \, v_j$$

where $K_{ij} = e^{-C_{ij}/\varepsilon}$ is the **Gibbs kernel**, and $u \in \mathbb{R}^n_+$, $v \in \mathbb{R}^m_+$ are scaling vectors.

Enforcing the marginal constraints $\pi \mathbf{1} = \mathbf{a}$ and $\pi^\top \mathbf{1} = \mathbf{b}$ leads to the **Sinkhorn algorithm** -- alternating row and column scaling:

$$\boxed{u^{(\ell+1)} = \frac{\mathbf{a}}{K v^{(\ell)}}, \qquad v^{(\ell+1)} = \frac{\mathbf{b}}{K^\top u^{(\ell+1)}}}$$

This converges linearly to the unique optimal $\pi^*_\varepsilon$, and as $\varepsilon \to 0$, $\pi^*_\varepsilon \to \pi^*$ (the unregularized optimum).

In [ ]:
# =============================================================================
# Section 5: Sinkhorn Algorithm
# =============================================================================

def sinkhorn(C, a, b, epsilon, max_iter=SINKHORN_MAX_ITER, tol=SINKHORN_TOL):
    """Sinkhorn algorithm for entropy-regularized optimal transport.

    Solves: min_pi <C, pi> - epsilon * H(pi)  s.t.  pi in Pi(a, b).
    Uses alternating Bregman projections (row/column scaling).

    Args:
        C: Cost matrix. Shape: (n, m).
        a: Source distribution. Shape: (n,).
        b: Target distribution. Shape: (m,).
        epsilon: Regularization strength. Scalar.
        max_iter: Maximum number of Sinkhorn iterations. Integer.
        tol: Convergence tolerance on marginal error. Scalar.

    Returns:
        pi: Approximate optimal transport plan. Shape: (n, m).
        cost: Transport cost <C, pi>. Scalar.
        log: Dictionary with convergence info:
            - 'marginal_errors': list of max marginal violations.
            - 'costs': list of transport costs at each iteration.
            - 'n_iter': number of iterations.
    """
    n, m = C.shape

    # Gibbs kernel
    K = np.exp(-C / epsilon)

    # Initialize scaling vectors
    u = np.ones(n)
    v = np.ones(m)

    marginal_errors = []
    costs = []

    for it in range(max_iter):
        # Row scaling: enforce pi @ 1 = a
        Kv = K @ v
        u = a / (Kv + 1e-300)

        # Column scaling: enforce pi^T @ 1 = b
        Ktu = K.T @ u
        v = b / (Ktu + 1e-300)

        # Compute transport plan
        pi = u[:, None] * K * v[None, :]

        # Marginal errors
        row_err = np.max(np.abs(pi.sum(axis=1) - a))
        col_err = np.max(np.abs(pi.sum(axis=0) - b))
        marg_err = max(row_err, col_err)
        marginal_errors.append(marg_err)

        # Cost
        cost = np.sum(C * pi)
        costs.append(cost)

        if marg_err < tol:
            break

    log = {
        'marginal_errors': marginal_errors,
        'costs': costs,
        'n_iter': it + 1,
    }

    return pi, cost, log

In [ ]:
# =============================================================================
# Run Sinkhorn for Multiple Epsilon Values
# =============================================================================

sinkhorn_results = {}
print(f"{'epsilon':>10s} | {'Sinkhorn cost':>14s} | {'LP cost':>10s} | {'Rel. diff':>10s} | {'Iters':>6s} | {'Marginal err':>12s}")
print('-' * 80)

for eps in EPSILON_VALUES:
    pi_sink, cost_sink, log_sink = sinkhorn(C, a, b, eps)
    sinkhorn_results[eps] = (pi_sink, cost_sink, log_sink)
    rel_diff = np.abs(cost_sink - ot_cost_lp) / ot_cost_lp
    print(f"{eps:10.4f} | {cost_sink:14.6f} | {ot_cost_lp:10.6f} | {rel_diff:10.2e} | {log_sink['n_iter']:6d} | {log_sink['marginal_errors'][-1]:12.2e}")

In [ ]:
# =============================================================================
# Verification: Sinkhorn Marginals
# =============================================================================

# Use the smallest epsilon for strictest check
eps_check = 0.01
pi_check = sinkhorn_results[eps_check][0]

row_err = np.max(np.abs(pi_check.sum(axis=1) - a))
col_err = np.max(np.abs(pi_check.sum(axis=0) - b))
marg_err = max(row_err, col_err)
status_marg = "PASS" if marg_err < MARGINAL_TOL else "FAIL"

print(f"Sinkhorn marginals match source/target (eps={eps_check}): max relative error = {marg_err:.2e} [{status_marg}]")
print(f"  Row marginal max error: {row_err:.2e}")
print(f"  Col marginal max error: {col_err:.2e}")

In [ ]:
# =============================================================================
# 2x2 Panel: Sinkhorn Analysis
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: transport plans for different epsilon
ax = axes[0, 0]
eps_show = [1.0, 0.1, 0.01]
for idx, eps in enumerate(eps_show):
    pi_s = sinkhorn_results[eps][0]
    offset = idx * (N_TARGET + 1)
    for i in range(N_SOURCE):
        for j in range(N_TARGET):
            if pi_s[i, j] > 0.005:
                ax.plot([i, j + offset + N_SOURCE + 1], [1, 0],
                        color=plt.cm.viridis(idx / 3),
                        alpha=min(pi_s[i, j] * 10, 1.0),
                        linewidth=pi_s[i, j] * 20)
ax.set_title('Transport Plans: $\\varepsilon$ = ' + ', '.join([str(e) for e in eps_show]))
ax.set_ylim(-0.2, 1.3)

# Top-right: Sinkhorn convergence (marginal error vs iterations)
ax = axes[0, 1]
colors_eps = [PRIMARY, SECONDARY, TERTIARY, ACCENT]
for idx, eps in enumerate(EPSILON_VALUES):
    log = sinkhorn_results[eps][2]
    ax.semilogy(log['marginal_errors'], color=colors_eps[idx], linewidth=2,
                label=f'$\\varepsilon$ = {eps}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Max Marginal Error')
ax.set_title('Sinkhorn Convergence')
ax.legend()

# Bottom-left: cost vs epsilon
ax = axes[1, 0]
eps_range = np.logspace(-3, 1, 30)
costs_vs_eps = []
for eps in eps_range:
    _, cost_e, _ = sinkhorn(C, a, b, eps, max_iter=2000)
    costs_vs_eps.append(cost_e)
ax.semilogx(eps_range, costs_vs_eps, 'o-', color=PRIMARY, markersize=4)
ax.axhline(ot_cost_lp, color=SECONDARY, linestyle='--', linewidth=2,
           label=f'LP optimal = {ot_cost_lp:.4f}')
ax.set_xlabel('$\\varepsilon$')
ax.set_ylabel('Transport Cost')
ax.set_title('Sinkhorn Cost Converges to LP as $\\varepsilon \\to 0$')
ax.legend()

# Bottom-right: heatmaps of transport plans
ax = axes[1, 1]
# Show the epsilon=0.01 plan vs LP plan
combined = np.zeros((N_SOURCE, 2 * N_TARGET + 1))
combined[:, :N_TARGET] = pi_lp
combined[:, N_TARGET + 1:] = sinkhorn_results[0.01][0]
im = ax.imshow(combined, cmap='Blues', aspect='auto')
ax.axvline(N_TARGET - 0.5, color='red', linewidth=2)
ax.set_title('LP (left) vs Sinkhorn $\\varepsilon$=0.01 (right)')
ax.set_xlabel('Target index')
ax.set_ylabel('Source index')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

---
# Section 6: Application -- Multi-Robot Task Allocation

## Setup

Consider $N$ robots at positions $\{r_i\}$ that must be assigned to $N$ task locations $\{t_j\}$. Each robot must go to exactly one task, and each task must be served by exactly one robot.

This is an **assignment problem** -- a special case of OT where both distributions are uniform with equal support sizes. The cost is the squared Euclidean distance: $C_{ij} = \|r_i - t_j\|^2$.

We compare:
1. **Sinkhorn** (entropic OT, approximate)
2. **Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`, exact)

In [ ]:
# =============================================================================
# Section 6: Multi-Robot Task Allocation
# =============================================================================

def task_allocation_ot(robot_pos, task_pos, epsilon=0.01):
    """Solve multi-robot task allocation using optimal transport.

    Compares Sinkhorn (entropic OT) with the Hungarian algorithm (exact).

    Args:
        robot_pos: Robot positions. Shape: (N, 2).
        task_pos: Task positions. Shape: (N, 2).
        epsilon: Sinkhorn regularization. Scalar.

    Returns:
        results: Dictionary with keys:
            - 'sinkhorn_plan': Sinkhorn transport plan. Shape: (N, N).
            - 'sinkhorn_cost': Sinkhorn assignment cost. Scalar.
            - 'sinkhorn_assignment': Hard assignment from Sinkhorn. Shape: (N,).
            - 'hungarian_assignment': Hungarian assignment. Shape: (N,).
            - 'hungarian_cost': Hungarian assignment cost. Scalar.
            - 'C': Cost matrix. Shape: (N, N).
    """
    N = len(robot_pos)
    assert len(task_pos) == N

    # Cost matrix: squared Euclidean distance
    C_alloc = cdist(robot_pos, task_pos, metric='sqeuclidean')

    # Uniform distributions
    a_alloc = np.ones(N) / N
    b_alloc = np.ones(N) / N

    # Sinkhorn
    pi_sink, cost_sink, log_sink = sinkhorn(C_alloc, a_alloc, b_alloc, epsilon)
    # Hard assignment from Sinkhorn: each robot goes to its highest-weight target
    sink_assignment = np.argmax(pi_sink, axis=1)

    # Hungarian algorithm (exact)
    row_ind, col_ind = linear_sum_assignment(C_alloc)
    hungarian_cost = C_alloc[row_ind, col_ind].sum() / N  # Normalize by N for comparison

    results = {
        'sinkhorn_plan': pi_sink,
        'sinkhorn_cost': cost_sink,
        'sinkhorn_assignment': sink_assignment,
        'hungarian_assignment': col_ind,
        'hungarian_cost': hungarian_cost,
        'C': C_alloc,
        'log': log_sink,
    }

    return results

In [ ]:
# =============================================================================
# Generate Robot and Task Positions
# =============================================================================

np.random.seed(42)

# Robots clustered on the left, tasks on the right
robot_pos = np.random.randn(N_ROBOTS, 2) * 1.5 + np.array([2.0, 5.0])
task_pos = np.random.randn(N_TASKS, 2) * 1.5 + np.array([8.0, 5.0])

# Run allocation
alloc_results = task_allocation_ot(robot_pos, task_pos, epsilon=0.01)

print(f"Sinkhorn cost:    {alloc_results['sinkhorn_cost']:.4f}")
print(f"Hungarian cost:   {alloc_results['hungarian_cost']:.4f}")
print(f"\nSinkhorn assignment:  {alloc_results['sinkhorn_assignment']}")
print(f"Hungarian assignment: {alloc_results['hungarian_assignment']}")
match = np.all(alloc_results['sinkhorn_assignment'] == alloc_results['hungarian_assignment'])
print(f"Assignments match:    {match}")

In [ ]:
# =============================================================================
# 2-Panel: Robot Task Allocation Visualization
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (ax, title, assignment) in enumerate(zip(
    axes,
    ['Sinkhorn Assignment', 'Hungarian Assignment (Exact)'],
    [alloc_results['sinkhorn_assignment'], alloc_results['hungarian_assignment']]
)):
    # Plot robots
    ax.scatter(robot_pos[:, 0], robot_pos[:, 1], s=200, c=PRIMARY,
              marker='o', zorder=5, edgecolors='black', linewidth=1,
              label='Robots')
    # Plot tasks
    ax.scatter(task_pos[:, 0], task_pos[:, 1], s=200, c=SECONDARY,
              marker='s', zorder=5, edgecolors='black', linewidth=1,
              label='Tasks')

    # Draw assignment arrows
    total_cost = 0
    for i in range(N_ROBOTS):
        j = assignment[i]
        dx = task_pos[j, 0] - robot_pos[i, 0]
        dy = task_pos[j, 1] - robot_pos[i, 1]
        dist_sq = dx**2 + dy**2
        total_cost += dist_sq
        ax.annotate('', xy=(task_pos[j, 0], task_pos[j, 1]),
                    xytext=(robot_pos[i, 0], robot_pos[i, 1]),
                    arrowprops=dict(arrowstyle='->', color=TERTIARY, lw=2))

    # Label robots and tasks
    for i in range(N_ROBOTS):
        ax.annotate(f'R{i}', (robot_pos[i, 0], robot_pos[i, 1]),
                    textcoords="offset points", xytext=(0, 12),
                    ha='center', fontsize=9, fontweight='bold')
    for j in range(N_TASKS):
        ax.annotate(f'T{j}', (task_pos[j, 0], task_pos[j, 1]),
                    textcoords="offset points", xytext=(0, 12),
                    ha='center', fontsize=9, fontweight='bold')

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\n(Total $\\|\\cdot\\|^2$ cost = {total_cost / N_ROBOTS:.2f})')
    ax.legend(loc='lower left')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
# Section 7: Wasserstein Barycenters

## Definition

The **Wasserstein barycenter** of distributions $\{\nu_k\}_{k=1}^K$ with weights $\{\lambda_k\}$ is the distribution that minimizes the weighted sum of Wasserstein distances:

$$\boxed{\bar{\nu} = \arg\min_{\mu} \sum_{k=1}^K \lambda_k \, W_2^2(\mu, \nu_k)}$$

This is the **Fr\'echet mean** in Wasserstein space. Unlike Euclidean averaging of densities, Wasserstein barycenters perform **displacement interpolation** -- they deform the geometry rather than blending intensities.

## Fixed-Point Iteration (Sinkhorn Barycenter)

For distributions on a fixed support grid, the Wasserstein barycenter with entropic regularization can be computed via a fixed-point iteration that alternates between:
1. Computing optimal transport plans from the current barycenter to each $\nu_k$
2. Updating the barycenter using the geometric mean of the transport marginals

In [ ]:
# =============================================================================
# Section 7: Wasserstein Barycenter
# =============================================================================

def wasserstein_barycenter(distributions, support_points, weights=None,
                           epsilon=0.05, max_iter=BARYCENTER_MAX_ITER,
                           tol=BARYCENTER_TOL):
    """Compute the entropy-regularized Wasserstein barycenter via fixed-point Sinkhorn.

    All distributions share the same support grid. The barycenter is computed
    as the geometric mean of the row-scaling vectors from Sinkhorn iterations.

    Args:
        distributions: List of K distributions, each Shape: (N,).
        support_points: Common support grid. Shape: (N,) or (N, d).
        weights: Barycentric weights. Shape: (K,). Default: uniform.
        epsilon: Sinkhorn regularization. Scalar.
        max_iter: Maximum fixed-point iterations. Integer.
        tol: Convergence tolerance. Scalar.

    Returns:
        barycenter: The Wasserstein barycenter. Shape: (N,).
        history: List of barycenter iterates for convergence monitoring.
    """
    K = len(distributions)
    N = len(distributions[0])

    if weights is None:
        weights = np.ones(K) / K

    # Compute cost matrix (squared Euclidean on the support grid)
    if support_points.ndim == 1:
        C_bary = (support_points[:, None] - support_points[None, :]) ** 2
    else:
        C_bary = cdist(support_points, support_points, metric='sqeuclidean')

    # Gibbs kernel
    K_mat = np.exp(-C_bary / epsilon)

    # Initialize barycenter as uniform
    bary = np.ones(N) / N
    history = [bary.copy()]

    # Initialize v vectors for each distribution
    v_list = [np.ones(N) for _ in range(K)]

    for it in range(max_iter):
        bary_old = bary.copy()

        # Compute u vectors: u_k = bary / (K @ v_k)
        u_list = []
        for k in range(K):
            Kv = K_mat @ v_list[k]
            u_k = bary / (Kv + 1e-300)
            u_list.append(u_k)

        # Update v vectors: v_k = nu_k / (K^T @ u_k)
        for k in range(K):
            Ktu = K_mat.T @ u_list[k]
            v_list[k] = distributions[k] / (Ktu + 1e-300)

        # Update barycenter: geometric weighted mean of (u_k * K @ v_k)
        log_bary = np.zeros(N)
        for k in range(K):
            Kv = K_mat @ v_list[k]
            transport_marginal = u_list[k] * Kv
            log_bary += weights[k] * np.log(transport_marginal + 1e-300)
        bary = np.exp(log_bary)
        bary = bary / bary.sum()  # Normalize

        history.append(bary.copy())

        # Check convergence
        change = np.max(np.abs(bary - bary_old))
        if change < tol:
            break

    return bary, history

In [ ]:
# =============================================================================
# Create Test Distributions for Barycenter
# =============================================================================

# Common support grid
support = np.linspace(-5, 10, N_SUPPORT)

# Three Gaussian-like distributions
def make_discrete_gaussian(support, mu, sigma):
    """Create a discrete Gaussian distribution on the given support.

    Args:
        support: Support points. Shape: (N,).
        mu: Mean. Scalar.
        sigma: Standard deviation. Scalar.

    Returns:
        p: Discrete Gaussian weights. Shape: (N,).
    """
    p = np.exp(-0.5 * ((support - mu) / sigma) ** 2)
    return p / p.sum()

nu_1 = make_discrete_gaussian(support, -2.0, 0.8)
nu_2 = make_discrete_gaussian(support, 3.0, 1.2)
nu_3 = make_discrete_gaussian(support, 7.0, 0.6)

distributions = [nu_1, nu_2, nu_3]
dist_names = ['$\\nu_1$ ($\\mu$=-2)', '$\\nu_2$ ($\\mu$=3)', '$\\nu_3$ ($\\mu$=7)']

In [ ]:
# =============================================================================
# Compute Wasserstein Barycenter
# =============================================================================

# Uniform weights
bary_uniform, bary_hist = wasserstein_barycenter(
    distributions, support, weights=np.array([1/3, 1/3, 1/3]),
    epsilon=0.05, max_iter=BARYCENTER_MAX_ITER
)

# Euclidean mean for comparison
euclidean_mean = (nu_1 + nu_2 + nu_3) / 3.0

print(f"Barycenter converged in {len(bary_hist) - 1} iterations")
print(f"Barycenter sum: {bary_uniform.sum():.6f} (should be 1.0)")
print(f"Barycenter peak at x = {support[np.argmax(bary_uniform)]:.2f}")

In [ ]:
# =============================================================================
# Verification: Barycenter Symmetry
# =============================================================================

# For two symmetric distributions placed at +d and -d, the barycenter
# should be symmetric about 0.
support_sym = np.linspace(-8, 8, N_SUPPORT)
nu_left = make_discrete_gaussian(support_sym, -3.0, 1.0)
nu_right = make_discrete_gaussian(support_sym, 3.0, 1.0)

bary_sym, _ = wasserstein_barycenter(
    [nu_left, nu_right], support_sym, weights=np.array([0.5, 0.5]),
    epsilon=0.05, max_iter=BARYCENTER_MAX_ITER
)

# The barycenter should be symmetric: bary(x) = bary(-x)
# Since support is symmetric about 0, bary[i] should equal bary[N-1-i]
symmetry_err = np.max(np.abs(bary_sym - bary_sym[::-1]))
status_sym = "PASS" if symmetry_err < SYMMETRY_TOL else "FAIL"
print(f"Barycenter is symmetric for symmetric inputs: max relative error = {symmetry_err:.2e} [{status_sym}]")

# The center of mass should be at 0
com = np.sum(support_sym * bary_sym)
print(f"Center of mass: {com:.6f} (should be ~0)")

In [ ]:
# =============================================================================
# 2-Panel: Barycenter Visualization
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Wasserstein barycenter vs Euclidean mean
ax = axes[0]
ax.plot(support, nu_1, color=PRIMARY, linewidth=1.5, alpha=0.6, label=dist_names[0])
ax.plot(support, nu_2, color=SECONDARY, linewidth=1.5, alpha=0.6, label=dist_names[1])
ax.plot(support, nu_3, color=TERTIARY, linewidth=1.5, alpha=0.6, label=dist_names[2])
ax.plot(support, bary_uniform, color='black', linewidth=2.5, label='Wasserstein barycenter')
ax.plot(support, euclidean_mean, color=ACCENT, linewidth=2, linestyle='--',
        label='Euclidean mean')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.set_title('Wasserstein Barycenter vs Euclidean Mean')
ax.legend(fontsize=10)

# Right: Displacement interpolation (varying weights between nu_1 and nu_2)
ax = axes[1]
n_interp = 7
interp_weights = np.linspace(0, 1, n_interp)
colors_interp = plt.cm.viridis(np.linspace(0, 1, n_interp))

for idx, alpha in enumerate(interp_weights):
    bary_interp, _ = wasserstein_barycenter(
        [nu_1, nu_2], support,
        weights=np.array([1 - alpha, alpha]),
        epsilon=0.05, max_iter=BARYCENTER_MAX_ITER
    )
    ax.plot(support, bary_interp + idx * 0.005, color=colors_interp[idx],
            linewidth=2, label=f'$\\alpha$ = {alpha:.2f}')

ax.set_xlabel('x')
ax.set_ylabel('Density (stacked)')
ax.set_title('Displacement Interpolation: $\\nu_1 \\to \\nu_2$')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

---
# Section 8: Comprehensive Visualizations

In this section we produce the key visualizations that bring the theory to life: transport plan heatmaps, optimal flow arrows, Sinkhorn convergence diagnostics, the Wasserstein distance matrix, and the barycentric interpolation sequence.

In [ ]:
# =============================================================================
# Visualization 1: Transport Plan Heatmap with Marginals
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plans = [
    (pi_lp, 'LP Optimal Plan'),
    (sinkhorn_results[0.1][0], 'Sinkhorn ($\\varepsilon$=0.1)'),
    (sinkhorn_results[0.01][0], 'Sinkhorn ($\\varepsilon$=0.01)'),
]

for ax, (plan, title) in zip(axes, plans):
    im = ax.imshow(plan, cmap='Blues', aspect='auto', vmin=0)
    # Annotate nonzero entries
    for i in range(plan.shape[0]):
        for j in range(plan.shape[1]):
            if plan[i, j] > 0.005:
                ax.text(j, i, f'{plan[i,j]:.3f}', ha='center', va='center',
                        fontsize=8, color='red' if plan[i,j] > 0.05 else 'gray')
    ax.set_xlabel('Target j')
    ax.set_ylabel('Source i')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Visualization 2: Optimal Flow Arrows (Source -> Target)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Plot source and target points
ax.scatter(x_source[:, 0], x_source[:, 1], s=a * 2000, c=PRIMARY,
          marker='o', zorder=5, edgecolors='black', linewidth=1.5,
          label='Source')
ax.scatter(y_target[:, 0], y_target[:, 1], s=b * 2000, c=SECONDARY,
          marker='s', zorder=5, edgecolors='black', linewidth=1.5,
          label='Target')

# Draw transport flows (LP optimal)
max_flow = pi_lp.max()
for i in range(N_SOURCE):
    for j in range(N_TARGET):
        if pi_lp[i, j] > 1e-6:
            flow = pi_lp[i, j]
            ax.annotate('',
                       xy=(y_target[j, 0], y_target[j, 1]),
                       xytext=(x_source[i, 0], x_source[i, 1]),
                       arrowprops=dict(
                           arrowstyle='->', color=TERTIARY,
                           lw=flow / max_flow * 5 + 0.5,
                           alpha=0.8,
                       ))
            # Label flow amount at midpoint
            mid_x = (x_source[i, 0] + y_target[j, 0]) / 2
            mid_y = (x_source[i, 1] + y_target[j, 1]) / 2
            ax.text(mid_x, mid_y, f'{flow:.3f}', fontsize=8,
                   ha='center', va='center',
                   bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.8))

# Label points
for i in range(N_SOURCE):
    ax.annotate(f'$x_{i}$ ({a[i]:.2f})', (x_source[i, 0], x_source[i, 1]),
               textcoords="offset points", xytext=(-15, 12),
               fontsize=9, fontweight='bold')
for j in range(N_TARGET):
    ax.annotate(f'$y_{j}$ ({b[j]:.2f})', (y_target[j, 0], y_target[j, 1]),
               textcoords="offset points", xytext=(10, 10),
               fontsize=9, fontweight='bold')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Optimal Transport Flow (LP, cost = {ot_cost_lp:.4f})')
ax.legend(loc='upper left', fontsize=12)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Visualization 3: Wasserstein Distance Matrix Between Multiple Distributions
# =============================================================================

# Create several 1D distributions
np.random.seed(42)
support_grid = np.linspace(-5, 10, 40)

dist_collection = [
    make_discrete_gaussian(support_grid, -2.0, 1.0),
    make_discrete_gaussian(support_grid, 0.0, 0.5),
    make_discrete_gaussian(support_grid, 2.0, 1.5),
    make_discrete_gaussian(support_grid, 5.0, 0.8),
    make_discrete_gaussian(support_grid, 7.0, 1.2),
]
dist_labels = ['N(-2,1)', 'N(0,0.5)', 'N(2,1.5)', 'N(5,0.8)', 'N(7,1.2)']
n_dists = len(dist_collection)

# Compute pairwise W2 distances
C_grid = (support_grid[:, None] - support_grid[None, :]) ** 2
W2_matrix = np.zeros((n_dists, n_dists))

for i in range(n_dists):
    for j in range(i + 1, n_dists):
        pi_ij, cost_ij = kantorovich_lp(C_grid, dist_collection[i], dist_collection[j])
        W2_matrix[i, j] = np.sqrt(cost_ij)
        W2_matrix[j, i] = W2_matrix[i, j]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distributions
ax = axes[0]
colors_dist = [PRIMARY, SECONDARY, TERTIARY, ACCENT, 'purple']
for idx, (d, lbl) in enumerate(zip(dist_collection, dist_labels)):
    ax.plot(support_grid, d, color=colors_dist[idx], linewidth=2, label=lbl)
ax.set_xlabel('x')
ax.set_ylabel('Weight')
ax.set_title('Collection of Distributions')
ax.legend(fontsize=10)

# Right: W2 distance matrix heatmap
ax = axes[1]
im = ax.imshow(W2_matrix, cmap='viridis', aspect='auto')
for i in range(n_dists):
    for j in range(n_dists):
        ax.text(j, i, f'{W2_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=10, color='white' if W2_matrix[i,j] > W2_matrix.max() * 0.5 else 'black')
ax.set_xticks(range(n_dists))
ax.set_yticks(range(n_dists))
ax.set_xticklabels(dist_labels, fontsize=9, rotation=45)
ax.set_yticklabels(dist_labels, fontsize=9)
ax.set_title('$W_2$ Distance Matrix')
plt.colorbar(im, ax=ax, label='$W_2$ distance')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Visualization 4: Barycentric Interpolation Sequence
# =============================================================================

# Create a smooth interpolation sequence between three distributions
# using barycentric coordinates on a triangle

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Row 1: interpolation from nu_1 to nu_2
n_steps = 5
alphas_12 = np.linspace(0, 1, n_steps)
for idx, alpha in enumerate(alphas_12):
    ax = axes[0, idx]
    bary_12, _ = wasserstein_barycenter(
        [nu_1, nu_2], support,
        weights=np.array([1 - alpha, alpha]),
        epsilon=0.05, max_iter=BARYCENTER_MAX_ITER
    )
    ax.fill_between(support, 0, bary_12, alpha=0.3, color=PRIMARY)
    ax.plot(support, bary_12, color=PRIMARY, linewidth=2)
    ax.plot(support, nu_1, color=SECONDARY, linewidth=1, alpha=0.4, linestyle='--')
    ax.plot(support, nu_2, color=TERTIARY, linewidth=1, alpha=0.4, linestyle='--')
    ax.set_title(f'$\\alpha$ = {alpha:.2f}', fontsize=11)
    ax.set_ylim(0, max(nu_1.max(), nu_2.max()) * 1.2)
    ax.set_xlabel('x')
    if idx == 0:
        ax.set_ylabel('$\\nu_1 \\to \\nu_2$')

# Row 2: interpolation from nu_1 to nu_3
alphas_13 = np.linspace(0, 1, n_steps)
for idx, alpha in enumerate(alphas_13):
    ax = axes[1, idx]
    bary_13, _ = wasserstein_barycenter(
        [nu_1, nu_3], support,
        weights=np.array([1 - alpha, alpha]),
        epsilon=0.05, max_iter=BARYCENTER_MAX_ITER
    )
    ax.fill_between(support, 0, bary_13, alpha=0.3, color=ACCENT)
    ax.plot(support, bary_13, color=ACCENT, linewidth=2)
    ax.plot(support, nu_1, color=SECONDARY, linewidth=1, alpha=0.4, linestyle='--')
    ax.plot(support, nu_3, color=TERTIARY, linewidth=1, alpha=0.4, linestyle='--')
    ax.set_title(f'$\\alpha$ = {alpha:.2f}', fontsize=11)
    ax.set_ylim(0, max(nu_1.max(), nu_3.max()) * 1.2)
    ax.set_xlabel('x')
    if idx == 0:
        ax.set_ylabel('$\\nu_1 \\to \\nu_3$')

plt.suptitle('Displacement Interpolation (Wasserstein Geodesics)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Visualization 5: Sinkhorn Convergence Deep Dive
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: marginal error vs iteration for multiple epsilon
ax = axes[0]
colors_eps = [PRIMARY, SECONDARY, TERTIARY, ACCENT]
for idx, eps in enumerate(EPSILON_VALUES):
    log = sinkhorn_results[eps][2]
    ax.semilogy(log['marginal_errors'], color=colors_eps[idx], linewidth=2,
                label=f'$\\varepsilon$ = {eps}')
ax.axhline(MARGINAL_TOL, color='gray', linestyle=':', linewidth=1, label=f'Tol = {MARGINAL_TOL:.0e}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Max Marginal Error')
ax.set_title('Sinkhorn: Marginal Error vs Iteration')
ax.legend()

# Middle: transport cost vs iteration
ax = axes[1]
for idx, eps in enumerate(EPSILON_VALUES):
    log = sinkhorn_results[eps][2]
    ax.plot(log['costs'], color=colors_eps[idx], linewidth=2,
            label=f'$\\varepsilon$ = {eps}')
ax.axhline(ot_cost_lp, color='black', linestyle='--', linewidth=2,
           label=f'LP optimal = {ot_cost_lp:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Transport Cost $\\langle C, \\pi \\rangle$')
ax.set_title('Sinkhorn: Cost vs Iteration')
ax.legend()

# Right: iterations needed vs epsilon
ax = axes[2]
eps_sweep = np.logspace(-3, 1, 20)
iters_needed = []
costs_at_conv = []
for eps in eps_sweep:
    _, cost_e, log_e = sinkhorn(C, a, b, eps, max_iter=5000, tol=1e-8)
    iters_needed.append(log_e['n_iter'])
    costs_at_conv.append(cost_e)

ax.semilogx(eps_sweep, iters_needed, 'o-', color=PRIMARY, markersize=5)
ax.set_xlabel('$\\varepsilon$')
ax.set_ylabel('Iterations to Converge')
ax.set_title('Sinkhorn: Computational Cost vs Regularization')

plt.tight_layout()
plt.show()

---
# Section 9: Summary of Verifications

In [ ]:
# =============================================================================
# Section 9: All Verifications Summary
# =============================================================================

print("=" * 70)
print("VERIFICATION SUMMARY")
print("=" * 70)

# 1. Primal-dual gap
gap = np.abs(primal_value - dual_value)
s1 = "PASS" if gap < DUALITY_GAP_TOL else "FAIL"
print(f"Primal-dual gap: max relative error = {gap:.2e} [{s1}]")

# 2. Sinkhorn marginals
pi_s01 = sinkhorn_results[0.01][0]
marg_err_s = max(np.max(np.abs(pi_s01.sum(axis=1) - a)),
                 np.max(np.abs(pi_s01.sum(axis=0) - b)))
s2 = "PASS" if marg_err_s < MARGINAL_TOL else "FAIL"
print(f"Sinkhorn marginals match source/target: max relative error = {marg_err_s:.2e} [{s2}]")

# 3. 1D Wasserstein
err_1d = np.abs(W1_lp - W1_quantile) / W1_lp
s3 = "PASS" if err_1d < WASSERSTEIN_1D_TOL else "FAIL"
print(f"1D Wasserstein matches quantile formula: max relative error = {err_1d:.2e} [{s3}]")

# 4. Barycenter symmetry
s4 = "PASS" if symmetry_err < SYMMETRY_TOL else "FAIL"
print(f"Barycenter is symmetric for symmetric inputs: max relative error = {symmetry_err:.2e} [{s4}]")

print("=" * 70)
all_pass = all([s == "PASS" for s in [s1, s2, s3, s4]])
print(f"Overall: {'ALL TESTS PASSED' if all_pass else 'SOME TESTS FAILED'}")
print("=" * 70)

---
# Section 10: Extensions

## Gromov-Wasserstein Distance

When source and target live in **incomparable metric spaces** (different dimensions, different modalities), the standard Wasserstein distance is undefined. The **Gromov-Wasserstein (GW) distance** compares the *internal geometry* of each space:

$$\text{GW}(\mu, \nu) = \min_{\pi \in \Pi(\mu, \nu)} \sum_{i,j,k,l} |d_X(x_i, x_k) - d_Y(y_j, y_l)|^2 \, \pi_{ij} \pi_{kl}$$

This is a **quadratic** assignment problem (not linear), making it much harder. The entropic-regularized version can be solved via a projected gradient descent on the Sinkhorn iterates.

## Unbalanced Optimal Transport

Standard OT requires $\sum_i a_i = \sum_j b_j$. **Unbalanced OT** relaxes this by adding KL-divergence penalties on the marginals:

$$\min_{\pi \geq 0} \langle C, \pi \rangle + \rho_1 \text{KL}(\pi \mathbf{1} \| \mathbf{a}) + \rho_2 \text{KL}(\pi^\top \mathbf{1} \| \mathbf{b})$$

This is crucial for applications like comparing histograms of different total mass (e.g., unnormalized feature distributions).

## OT for Generative Models

Optimal transport has become central to generative modeling:

| Model | OT Connection |
|-------|---------------|
| **WGAN** | Discriminator approximates Kantorovich potential $f$ |
| **Sinkhorn divergence** | Debiased entropic OT as training loss |
| **Flow matching** | Conditional OT defines velocity field for continuous normalizing flows |
| **Schr\"odinger bridge** | Entropic OT between noise and data distributions |

## Computational OT at Scale

For large-scale problems ($n > 10^4$), several techniques are essential:

1. **Log-domain Sinkhorn:** Work with $\log u$, $\log v$ to avoid numerical overflow/underflow
2. **Multiscale Sinkhorn:** Coarse-to-fine hierarchy for faster convergence
3. **Sliced Wasserstein:** Average 1D OT over random projections -- $O(n \log n)$ per slice
4. **Stochastic OT:** Mini-batch approximations for very large sample sizes

### Key Takeaways

1. **Kantorovich's relaxation** transforms Monge's hard problem into a tractable LP
2. **Wasserstein distance** is a true metric on probability distributions that respects geometry
3. **Strong duality** connects primal (transport plan) and dual (Kantorovich potentials)
4. **Sinkhorn algorithm** makes OT computationally feasible via entropic regularization
5. **Wasserstein barycenters** perform displacement interpolation, preserving geometric structure
6. OT provides a principled framework for comparing distributions in machine learning, robotics, and beyond